# Convert Student Paper Index spreadsheet to Invenio json.

In [ ]:
import pandas as pd
import json
import os

In [ ]:
dir = '/Users/metadatagamechanger/ProjectsAndPlans/FieldStations/MooreaStudentPapers/'
input = dir + 'index_to_moorea_class_projects_1992-2022.csv'
input_df = pd.read_csv(input)
input_df.head()

## Create Invenio metadata records.
See https://inveniordm.docs.cern.ch/reference/metadata/ for reference

In [ ]:
for i in input_df.index:                                # loop input rows (one paper/metadata record per row)
    #
    # initialize invenio dictionary
    # 
    invenio_d = {
    "access": {
        "files": "public",
        "record": "public"
    },
    "files": {
        "enabled": True
    },
    "metadata": {}
    }
    invenio_d['metadata']['creators'] = []              # Add creators to metadata dictionary, initialize as empty list

    for p, person in enumerate(input_df.at[i, 'Authors, Primary'].split(';')):
        person = person.replace('Dr. ','')
        if ',' in person:
            toks = person.split(',')
        else:
            toks = person.split(' ,')

        if len(toks) < 2 or len(toks) > 4:
            continue

        family = toks[0].replace(',','') if ',' in person else toks[1]
        family = family.strip()

        given = " ".join(toks[1:]) if ',' in person else toks[0]
        given = given.strip()                                       # .replace(' ','+')

        invenio_d['metadata']['creators'].append({'person_or_org': {'family_name': family, 'given_name': given}, 'type': 'personal'})

    if len(str(input_df.at[i, 'Abstract'])) > 0 and str(input_df.at[i, 'Abstract']) != 'nan':
        invenio_d['metadata']['description'] = input_df.at[i, 'Abstract']

    identifier = f"{'_'.join([ x['person_or_org']['family_name'].replace(' ','')+'_'+ x['person_or_org']['given_name'].replace(' ','').replace('.','') for x in invenio_d['metadata']['creators'] ])}"
    identifier = identifier.replace('/','-').replace(':','-').replace('?','-').replace('"','-').replace('\'','-').replace(',','').replace('(','').replace(')','').replace('\n','_')
    identifier = '10.17913/bgtl/' + str(input_df.at[i, 'Pub Year']) + '/' + identifier
    invenio_d['metadata']['identifiers'] = [{'identifier': identifier, 'scheme': 'doi'}]

    invenio_d['metadata']['publication_date']   = input_df.at[i, 'Pub Year']
    invenio_d['metadata']['publisher']          = 'University of California, Berkeley'
    invenio_d['metadata']['resource_type']      = 'publication-preprint'

    contributor_Gump = {                                        # define Gump as a contributor
        "person_or_org": {
            "name": "Gump South Pacific Research Station",
            "type": "organizational",
            "identifiers": [{
            "scheme": "ror",
            "identifier": "04sk0et52"
            }],
        },
        "role": {"id": "sponsor"},
        "affiliations": [{
            "id": "04sk0et52",
            "name": "Gump South Pacific Research Station",
        }]
    }
    invenio_d['metadata']['contributors']      = [contributor_Gump]

    if len(str(input_df.at[i, 'Keywords'])) > 0 and str(input_df.at[i, 'Keywords']) != 'nan':
        keywords_l = str(input_df.at[i, 'Keywords']).split(';')              # some keywords are delimited with ";"
        if len(keywords_l) == 1:
            keywords_l = str(input_df.at[i, 'Keywords']).split(',')          # some keywords are delimited with ","

        for subject in keywords_l:
            subject = subject.strip()
            if len(subject) > 0:
                if 'subjects' not in invenio_d['metadata']:
                    invenio_d['metadata']['subjects'] = []
                invenio_d['metadata']['subjects'].append({'subject': subject})
    #
    # The index spreadsheet includes a column named "organism" that includes fairly high-level organism keywords
    # The metadata includes an element named subjectScheme that identifies keywords of a specific type.
    # In this case that is set to "organism" to differentiate these keywords from the general set
    #  
    if len(str(input_df.at[i, 'organism'])) > 0 and str(input_df.at[i, 'organism']) != 'nan':
        if 'subjects' not in invenio_d['metadata']:
            invenio_d['metadata']['subjects'] = []
        invenio_d['metadata']['subjects'].append({'subject': input_df.at[i, 'organism'], 'subjectScheme':'organism'})

    #
    # The index spreadsheet includes a column named "where" that includes fairly high-level location keywords
    # The metadata includes an element named subjectScheme that identifies keywords of a specific type.
    # In this case that is set to "where" to differentiate these keywords from the general set.
    #  
    if len(str(input_df.at[i, 'where'])) > 0 and str(input_df.at[i, 'where']) != 'nan':
        if 'subjects' not in invenio_d['metadata']:
            invenio_d['metadata']['subjects'] = []
        invenio_d['metadata']['subjects'].append({'subject': input_df.at[i, 'where'], 'subjectScheme':'where'})

    if len(str(input_df.at[i, 'Title Primary'])) > 0 and str(input_df.at[i, 'Title Primary']) != 'nan':
        invenio_d['metadata']['title'] = input_df.at[i, 'Title Primary']

    invenio_d['metadata']['version'] = 1

    #fileName = f"{'_'.join([ x['person_or_org']['family_name'].replace(' ','_') for x in invenio_d['metadata']['creators'] ])}_{invenio_d['metadata']['title'].replace(' ','_')}.json"
    fileName = f"{'_'.join([ x['person_or_org']['family_name'].replace(' ','')+'_'+ x['person_or_org']['given_name'].replace(' ','').replace('.','') for x in invenio_d['metadata']['creators'] ])}.json"
    fileName = fileName.replace('/','-').replace(':','-').replace('?','-').replace('"','-').replace('\'','-').replace(',','').replace('(','').replace(')','').replace('\n','_')
    output = dir + 'metadata/' + f'{str(invenio_d["metadata"]["publication_date"])}/' + fileName
    
    os.makedirs(os.path.dirname(output), exist_ok=True)
    print(f"writing metadata for {identifier} to {'/'.join(output.split('/')[-2:])}")
    with open(output, 'w') as f:
        json.dump(invenio_d, f, indent=2, default=str)
